# Module 14: Scaling with JAX
## Batch Agents and Hardware Acceleration

**Spinning Up in Active Inference** | Notebook 14 of 16

---

Active Inference involves repeated matrix multiplications, softmax operations, and belief propagation -- all of which are naturally parallelizable. JAX gives us three transformations that unlock massive speedups:

- **`jit`** -- compile a function once, run it fast forever
- **`vmap`** -- vectorize over a batch dimension (run 1000 agents as one matrix multiply)
- **`grad`** -- differentiate through the entire inference pipeline

In this module we will:
1. Build intuition for `jit`, `vmap`, and `grad` with simple examples
2. Use the `BatchAgent` class to run 1, 100, and 1000 agents simultaneously
3. Benchmark wall-clock time and see sub-linear scaling
4. Differentiate through free energy (which enables model learning)

**Prerequisites:** Modules 1-13, NumPy fluency. No JAX experience required.
**Time:** ~50 minutes.

In [ ]:
# -- Setup ------------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp

# PGMax AIF imports
from alf.generative_model import GenerativeModel
from alf.jax_native import (
    BatchAgent,
    jax_compute_efe_analytic,
    jax_evaluate_all_actions,
    jax_select_action,
)
from alf.free_energy import (
    jax_variational_free_energy,
    jax_expected_free_energy_decomposed,
    jax_efe_all_actions,
)
from alf.benchmarks.t_maze import (
    build_t_maze_model, TMazeEnv, run_t_maze,
    STATE_NAMES, OBS_NAMES, ACTION_NAMES,
)

import sys; sys.path.insert(0, '..')
from utils.plotting import COLORS

# Detect available devices
print(f"JAX version: {jax.__version__}")
print(f"Available devices: {jax.devices()}")
print(f"Default backend: {jax.default_backend()}")

## 14.1 JAX Primer: `jit` -- Compile Once, Run Fast

JAX's `jit` (Just-In-Time compilation) traces a Python function into XLA (Accelerated Linear Algebra) operations, then compiles to optimized machine code. The first call is slow (compilation), but subsequent calls are fast.

The key mental model: `jit` turns your Python function into a GPU/CPU kernel.

In [ ]:
# -- jit: compile a simple function -----------------------------------------

def softmax_numpy(x):
    """Standard softmax in NumPy."""
    e = np.exp(x - np.max(x))
    return e / e.sum()

def softmax_jax(x):
    """Standard softmax in JAX."""
    e = jnp.exp(x - jnp.max(x))
    return e / e.sum()

# JIT-compile the JAX version
softmax_jit = jax.jit(softmax_jax)

# Test data
x_np = np.random.randn(10000).astype(np.float32)
x_jax = jnp.array(x_np)

# Warm up JIT (first call includes compilation time)
_ = softmax_jit(x_jax).block_until_ready()

# Benchmark
n_trials = 100

t0 = time.perf_counter()
for _ in range(n_trials):
    _ = softmax_numpy(x_np)
t_numpy = (time.perf_counter() - t0) / n_trials

t0 = time.perf_counter()
for _ in range(n_trials):
    _ = softmax_jax(x_jax).block_until_ready()
t_jax = (time.perf_counter() - t0) / n_trials

t0 = time.perf_counter()
for _ in range(n_trials):
    _ = softmax_jit(x_jax).block_until_ready()
t_jit = (time.perf_counter() - t0) / n_trials

print(f"NumPy:       {t_numpy*1000:.3f} ms")
print(f"JAX (eager): {t_jax*1000:.3f} ms")
print(f"JAX (jit):   {t_jit*1000:.3f} ms")
print(f"\nJIT speedup over NumPy: {t_numpy/t_jit:.1f}x")

## 14.2 `vmap` -- Vectorize Over a Batch

`vmap` (vectorized map) automatically transforms a function that operates on a single element into one that operates on a batch. No loops needed -- JAX parallelizes across the batch dimension.

For Active Inference, this means: write your belief update for *one* agent, then `vmap` it to run 1000 agents in parallel.

In [ ]:
# -- vmap: vectorize belief updates over a batch of agents -------------------

def belief_update_single(prior, A_row):
    """Update beliefs for a single agent given one observation.

    Q(s) proportional to P(o|s) * P(s)
    """
    posterior = prior * A_row
    posterior = jnp.clip(posterior, 1e-16)
    return posterior / posterior.sum()

# Single agent
num_states = 4
prior_single = jnp.ones(num_states) / num_states
A_row_single = jnp.array([0.8, 0.1, 0.05, 0.05])  # P(o=0 | s)

posterior_single = belief_update_single(prior_single, A_row_single)
print("Single agent posterior:", posterior_single)

# Batch of 1000 agents with different priors
batch_size = 1000
key = jax.random.PRNGKey(42)
priors_batch = jax.random.dirichlet(key, jnp.ones(num_states), shape=(batch_size,))

# Different observation likelihoods for each agent
key2 = jax.random.PRNGKey(43)
A_rows_batch = jax.random.dirichlet(key2, jnp.ones(num_states), shape=(batch_size,))

# vmap: apply belief_update_single to each agent in the batch
belief_update_batch = jax.vmap(belief_update_single)

# JIT-compile the vmapped function for maximum speed
belief_update_batch_jit = jax.jit(belief_update_batch)

# Warm up
_ = belief_update_batch_jit(priors_batch, A_rows_batch).block_until_ready()

# Benchmark: loop vs vmap
t0 = time.perf_counter()
for _ in range(100):
    results_loop = []
    for i in range(batch_size):
        results_loop.append(belief_update_single(priors_batch[i], A_rows_batch[i]))
t_loop = (time.perf_counter() - t0) / 100

t0 = time.perf_counter()
for _ in range(100):
    _ = belief_update_batch_jit(priors_batch, A_rows_batch).block_until_ready()
t_vmap = (time.perf_counter() - t0) / 100

print(f"\nLoop ({batch_size} agents): {t_loop*1000:.2f} ms")
print(f"vmap ({batch_size} agents): {t_vmap*1000:.2f} ms")
print(f"Speedup: {t_loop/t_vmap:.1f}x")

## 14.3 `grad` -- Differentiate Through Free Energy

`jax.grad` computes exact gradients through arbitrary Python functions. For Active Inference, this lets us differentiate through:
- Variational Free Energy (for learning A/B matrices)
- Expected Free Energy (for meta-learning policy precision)

This is the key advantage of PGMax over pymdp: because our AIF computations are in JAX, we get gradients *for free*.

In [ ]:
# -- grad: differentiate Variational Free Energy ----------------------------

# Simple 4-state model
A_small = jnp.array([
    [0.9, 0.1, 0.1, 0.1],
    [0.1, 0.9, 0.1, 0.1],
    [0.0, 0.0, 0.8, 0.1],
    [0.0, 0.0, 0.0, 0.7],
])
# Normalize columns
A_small = A_small / A_small.sum(axis=0, keepdims=True)

prior_s = jnp.ones(4) / 4.0
observation = jnp.array(0)  # observed state 0

# Parameterize beliefs in unconstrained space (logits)
def vfe_from_logits(logits):
    """VFE as a function of unconstrained belief parameters."""
    q_s = jax.nn.softmax(logits)
    return jax_variational_free_energy(q_s, A_small, prior_s, observation)

# Compute gradient of VFE with respect to belief logits
grad_vfe = jax.grad(vfe_from_logits)

# Start from uniform beliefs
logits_init = jnp.zeros(4)

print("Gradient descent on Variational Free Energy:")
print(f"{'Step':>4} {'VFE':>10} {'Beliefs':>40}")
print("-" * 60)

logits = logits_init
lr = 0.5

for step in range(10):
    vfe_val = vfe_from_logits(logits)
    beliefs_current = jax.nn.softmax(logits)
    print(f"{step:>4} {float(vfe_val):>10.4f} {np.array(beliefs_current).round(4)}")

    # Gradient step: minimize VFE
    grads = grad_vfe(logits)
    logits = logits - lr * grads

# Final beliefs should concentrate on state 0 (matching observation)
final_beliefs = jax.nn.softmax(logits)
print(f"\nFinal beliefs: {np.array(final_beliefs).round(4)}")
print("Beliefs concentrate on state 0 -- the observed state.")
print("This is gradient-based inference: minimize VFE to find the posterior.")

## 14.4 The T-Maze with `BatchAgent`

Now let us put it all together. The `BatchAgent` class from `alf.jax_native` runs multiple Active Inference agents in parallel using `vmap` under the hood.

We start with a single agent on the T-maze to verify correctness, then scale up.

In [ ]:
# -- Build the T-maze model and create a BatchAgent -------------------------

gm_tmaze = build_t_maze_model()

print(f"T-maze model:")
print(f"  States:       {gm_tmaze.A[0].shape[1]} ({', '.join(STATE_NAMES)})")
print(f"  Observations: {gm_tmaze.A[0].shape[0]} ({', '.join(OBS_NAMES)})")
print(f"  Actions:      {gm_tmaze.B[0].shape[2]} ({', '.join(ACTION_NAMES)})")

# Single-agent BatchAgent
agent_1 = BatchAgent(gm_tmaze, batch_size=1, gamma=4.0, seed=42)

# Run one step: observe center position (obs=0)
observations = np.array([0])  # center observation
actions, info = agent_1.step_analytic(observations)

print(f"\nSingle agent step:")
print(f"  Observation: {OBS_NAMES[0]}")
print(f"  Action chosen: {ACTION_NAMES[int(actions[0])]}")
print(f"  Beliefs: {np.array(info['beliefs'][0][0]).round(3)}")
print(f"  EFE per action: {np.array(info['G'][0]).round(3)}")
print(f"  Policy probs: {np.array(info['policy_probs'][0]).round(3)}")

In [ ]:
# -- Scale up: 100 agents ---------------------------------------------------

agent_100 = BatchAgent(gm_tmaze, batch_size=100, gamma=4.0, seed=42)

# All agents observe center
obs_100 = np.zeros(100, dtype=int)
actions_100, info_100 = agent_100.step_analytic(obs_100)

# Distribution of chosen actions across 100 agents
action_counts = np.bincount(actions_100, minlength=len(ACTION_NAMES))
print("Action distribution across 100 agents:")
for i, name in enumerate(ACTION_NAMES):
    print(f"  {name}: {action_counts[i]} agents ({action_counts[i]}%)")

print(f"\nMean EFE per action: {np.mean(info_100['G'], axis=0).round(3)}")

In [ ]:
# -- Scale to 1000 agents ---------------------------------------------------

agent_1000 = BatchAgent(gm_tmaze, batch_size=1000, gamma=4.0, seed=42)

obs_1000 = np.zeros(1000, dtype=int)
actions_1000, info_1000 = agent_1000.step_analytic(obs_1000)

action_counts_1k = np.bincount(actions_1000, minlength=len(ACTION_NAMES))
print("Action distribution across 1000 agents:")
for i, name in enumerate(ACTION_NAMES):
    pct = action_counts_1k[i] / 10.0
    print(f"  {name}: {action_counts_1k[i]} agents ({pct:.1f}%)")

## 14.5 Benchmarking: Wall-Clock Scaling

The critical test: does running 1000 agents take 1000x longer than running 1 agent? With `vmap` and GPU acceleration, we expect **sub-linear scaling** -- the overhead of parallelization is amortized across the batch.

In [ ]:
# -- Benchmark: wall-clock time vs batch size --------------------------------

batch_sizes = [1, 10, 50, 100, 500, 1000]
times_per_step = []
n_warmup = 5
n_measure = 50

for bs in batch_sizes:
    agent = BatchAgent(gm_tmaze, batch_size=bs, gamma=4.0, seed=42)
    obs = np.zeros(bs, dtype=int)

    # Warm up (JIT compilation happens here)
    for _ in range(n_warmup):
        agent.reset()
        _ = agent.step_analytic(obs)

    # Measure
    t0 = time.perf_counter()
    for _ in range(n_measure):
        agent.reset()
        _ = agent.step_analytic(obs)
    elapsed = (time.perf_counter() - t0) / n_measure
    times_per_step.append(elapsed)

    per_agent = elapsed / bs * 1000  # ms per agent
    print(f"batch_size={bs:>5}  |  total: {elapsed*1000:.2f} ms  |  "
          f"per agent: {per_agent:.4f} ms")

# Plot scaling
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(batch_sizes, [t * 1000 for t in times_per_step], 'o-',
         color=COLORS.get('aif', '#4CAF50'), linewidth=2, markersize=8)
ax1.plot(batch_sizes, [times_per_step[0] * bs * 1000 for bs in batch_sizes],
         'k--', alpha=0.3, label='Linear scaling')
ax1.set_xlabel('Batch Size (number of agents)')
ax1.set_ylabel('Wall-clock time (ms)')
ax1.set_title('Total Time vs. Batch Size')
ax1.legend()
ax1.set_xscale('log')
ax1.set_yscale('log')

per_agent_times = [t / bs * 1000 for t, bs in zip(times_per_step, batch_sizes)]
ax2.plot(batch_sizes, per_agent_times, 'o-',
         color=COLORS.get('aif', '#4CAF50'), linewidth=2, markersize=8)
ax2.set_xlabel('Batch Size (number of agents)')
ax2.set_ylabel('Time per agent (ms)')
ax2.set_title('Per-Agent Cost vs. Batch Size\n(Lower = better parallelization)')
ax2.set_xscale('log')

plt.suptitle('JAX vmap Scaling: Sub-linear Cost Growth', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 14.6 JIT-Compiled EFE: `jax_compute_efe_analytic`

The `jax_compute_efe_analytic` function computes the Expected Free Energy for a single action using pure JAX operations. When JIT-compiled, it runs as a single fused kernel.

In [ ]:
# -- JIT-compiled EFE computation -------------------------------------------

A_jax = jnp.array(gm_tmaze.A[0])
B_jax = jnp.array(gm_tmaze.B[0])
C_jax = jnp.array(gm_tmaze.C[0])
beliefs_jax = jnp.array(gm_tmaze.D[0])

# Non-JIT version
efe_eager = jax_compute_efe_analytic(A_jax, B_jax[:, :, 0], C_jax, beliefs_jax)
print(f"EFE (eager, action=0): {float(efe_eager):.4f}")

# JIT-compiled version
efe_jit_fn = jax.jit(jax_compute_efe_analytic)
_ = efe_jit_fn(A_jax, B_jax[:, :, 0], C_jax, beliefs_jax).block_until_ready()  # warm up

efe_jit_val = efe_jit_fn(A_jax, B_jax[:, :, 0], C_jax, beliefs_jax)
print(f"EFE (jit,   action=0): {float(efe_jit_val):.4f}")
print(f"Match: {jnp.allclose(efe_eager, efe_jit_val)}")

# Benchmark
n_bench = 1000

t0 = time.perf_counter()
for _ in range(n_bench):
    _ = jax_compute_efe_analytic(A_jax, B_jax[:, :, 0], C_jax, beliefs_jax)
_ = _.block_until_ready()
t_eager = (time.perf_counter() - t0) / n_bench

t0 = time.perf_counter()
for _ in range(n_bench):
    _ = efe_jit_fn(A_jax, B_jax[:, :, 0], C_jax, beliefs_jax)
_ = _.block_until_ready()
t_jit = (time.perf_counter() - t0) / n_bench

print(f"\nEager: {t_eager*1e6:.1f} us")
print(f"JIT:   {t_jit*1e6:.1f} us")
print(f"JIT speedup: {t_eager/t_jit:.1f}x")

In [ ]:
# -- jax_evaluate_all_actions: vmap over the action dimension ----------------

# Evaluate EFE for ALL actions at once using vmap
G_all = jax_evaluate_all_actions(A_jax, B_jax, C_jax, beliefs_jax)

print("EFE for all actions (vmapped):")
for i, name in enumerate(ACTION_NAMES):
    print(f"  {name}: G = {float(G_all[i]):.4f} (lower = preferred)")

best_action = int(jnp.argmin(G_all))
print(f"\nBest action: {ACTION_NAMES[best_action]} (lowest EFE)")

# Compare with the decomposed version
G_decomp, prag_decomp, epist_decomp = jax_efe_all_actions(
    A_jax, B_jax, C_jax, beliefs_jax
)
print("\nDecomposition (vmapped):")
print(f"{'Action':<12} {'G_total':>10} {'Pragmatic':>10} {'Epistemic':>10}")
print("-" * 44)
for i, name in enumerate(ACTION_NAMES):
    print(f"{name:<12} {float(G_decomp[i]):>10.4f} {float(prag_decomp[i]):>10.4f} "
          f"{float(epist_decomp[i]):>10.4f}")

## 14.7 Inspecting the Computation Graph with `make_jaxpr`

`jax.make_jaxpr` shows the intermediate representation (IR) of a traced function. This reveals the computation graph that XLA compiles -- useful for understanding what JAX actually does and identifying potential bottlenecks.

In [ ]:
# -- Profile with make_jaxpr ------------------------------------------------

# Trace the EFE computation
jaxpr = jax.make_jaxpr(jax_compute_efe_analytic)(
    A_jax, B_jax[:, :, 0], C_jax, beliefs_jax
)

print("JAXPR for jax_compute_efe_analytic:")
print("=" * 60)
print(jaxpr)
print("=" * 60)

# Count operations
eqns = jaxpr.jaxpr.eqns
op_counts = {}
for eq in eqns:
    name = str(eq.primitive.name)
    op_counts[name] = op_counts.get(name, 0) + 1

print(f"\nOperation counts ({len(eqns)} total ops):")
for op, count in sorted(op_counts.items(), key=lambda x: -x[1]):
    print(f"  {op}: {count}")

In [ ]:
# -- Also trace the vmapped version for comparison --------------------------

jaxpr_vmap = jax.make_jaxpr(jax_evaluate_all_actions)(
    A_jax, B_jax, C_jax, beliefs_jax
)

eqns_vmap = jaxpr_vmap.jaxpr.eqns
print(f"vmapped evaluate_all_actions: {len(eqns_vmap)} total ops")
print(f"(vs. {len(eqns)} ops for single-action version)")
print("\nvmap adds a batch dimension but does NOT multiply the number of ops.")
print("This is why it scales so well -- the same computation runs in parallel.")

## 14.8 Differentiating Through EFE: Gradient of Free Energy

The ultimate payoff of JAX: we can differentiate through the *entire* Active Inference pipeline. This means:
- **Learn A/B matrices** by gradient descent on VFE (Module 11)
- **Optimize precision gamma** by differentiating through policy selection
- **Meta-learn preferences C** from outcome data

In [ ]:
# -- Gradient of EFE with respect to preferences C --------------------------

def total_efe_from_C(C):
    """Total EFE summed over all actions, as a function of preferences."""
    G_all_local = jax_evaluate_all_actions(A_jax, B_jax, C, beliefs_jax)
    return jnp.sum(G_all_local)

grad_efe_wrt_C = jax.grad(total_efe_from_C)

dG_dC = grad_efe_wrt_C(C_jax)

print("Gradient of total EFE with respect to preferences C:")
for i, name in enumerate(OBS_NAMES):
    print(f"  dG/dC[{name}] = {float(dG_dC[i]):.4f}")

print("\nInterpretation: increasing preference for an observation")
print("decreases EFE (makes actions that lead to it more valuable).")
print("This is how you could META-LEARN preferences from outcome data.")

In [ ]:
# -- Gradient of EFE w.r.t. beliefs: sensitivity analysis -------------------

def min_efe_from_beliefs(beliefs):
    """Minimum EFE (best action's EFE) as a function of beliefs."""
    G_all_local = jax_evaluate_all_actions(A_jax, B_jax, C_jax, beliefs)
    return jnp.min(G_all_local)

grad_efe_wrt_beliefs = jax.grad(min_efe_from_beliefs)

dG_dbeliefs = grad_efe_wrt_beliefs(beliefs_jax)

print("Sensitivity of best-action EFE to belief changes:")
for i, name in enumerate(STATE_NAMES):
    print(f"  dG/dQ({name}) = {float(dG_dbeliefs[i]):.4f}")

print("\nLarge gradients indicate states where belief changes")
print("most affect the agent's decision quality.")

## 14.9 Exercise: Profile and Optimize

**Challenge:** Identify the computational bottleneck in batch Active Inference and suggest optimizations.

1. **Profile the BatchAgent**: use `time.perf_counter()` to measure `step_analytic` for different model sizes (vary `num_states`).
2. **Identify the bottleneck**: is it belief update, EFE computation, or action selection?
3. **Try `jax.jit` with `static_argnums`**: can you JIT-compile the entire `step_analytic` method?
4. **GPU vs. CPU**: if you have a GPU available, compare `jax.devices('gpu')` performance.

Use the skeleton below.

In [ ]:
# -- Exercise: profile BatchAgent bottlenecks --------------------------------

# Measure time for each component separately
batch_size_ex = 500
agent_ex = BatchAgent(gm_tmaze, batch_size=batch_size_ex, gamma=4.0, seed=42)
obs_ex = np.zeros(batch_size_ex, dtype=int)

# Warm up
_ = agent_ex.step_analytic(obs_ex)
agent_ex.reset()

# Profile: total step time
n_profile = 100
t0 = time.perf_counter()
for _ in range(n_profile):
    agent_ex.reset()
    _ = agent_ex.step_analytic(obs_ex)
t_total = (time.perf_counter() - t0) / n_profile

print(f"BatchAgent.step_analytic ({batch_size_ex} agents):")
print(f"  Total: {t_total*1000:.2f} ms")
print(f"  Per agent: {t_total/batch_size_ex*1e6:.1f} us")

# TODO: break down into components (belief update, EFE, action selection)
# Hint: extract the three stages from step_analytic and time them separately
print("\nExercise: decompose step_analytic into its three stages")
print("and identify which one dominates the wall-clock time.")

## 14.10 Summary

JAX transforms unlock three capabilities for Active Inference:

1. **`jit`**: Compile AIF computations to optimized machine code. The EFE computation runs as a single fused kernel, often 10-100x faster than eager execution.

2. **`vmap`**: Run thousands of agents in parallel with no code changes. Write your inference for one agent, `vmap` it for a population. Scaling is sub-linear thanks to hardware parallelism.

3. **`grad`**: Differentiate through the entire inference pipeline. This enables gradient-based learning of A/B matrices, precision optimization, and meta-learning of preferences -- capabilities that are unique to PGMax vs. pymdp.

The `BatchAgent` class wraps these transformations into a clean API for batch simulation. In the next module, we will use these tools for hierarchical models.

**Next:** [Module 15: Hierarchical Active Inference](15_hierarchical_aif.ipynb)